In [ ]:
# Cell 1: Install dependencies
!pip install -q scanpy anndata igraph leidenalg scikit-learn scipy cellxgene-census

In [ ]:
# Cell 2: Mount Drive and set paths
from google.colab import drive
drive.mount('/content/drive')

import os

VOCAB_DIR   = '/content/drive/MyDrive/CellJEPA_results/universal_vocab/'
MT_DIR      = '/content/drive/MyDrive/CellJEPA_results/multitissue_universal/'
RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/multitissue_transfer/'

for d in [VOCAB_DIR, MT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Upload all .py files to /content/ before running cells below:
# cell_jepa.py, cell_sigreg.py, losses.py, preprocessing.py, trainer.py,
# metrics.py, compare_pbmc3k.py, run_ablation.py,
# build_universal_vocab.py, pretrain_universal.py, run_transfer_universal.py

VOCAB_FILE = os.path.join(VOCAB_DIR, 'universal_gene_names.json')

print('Drive mounted.')
print('Vocab file:', VOCAB_FILE)
print('Pre-train output:', MT_DIR)
print('Results output:', RESULTS_DIR)

In [ ]:
# Cell 3: Build universal vocab — smoke test (~1 min, uses built-in datasets)
import subprocess, os

result = subprocess.run(
    ['python3', '-u', '/content/build_universal_vocab.py',
     '--smoke_test',
     '--drive_dir', '/content/vocab_smoke/'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# Cell 4: Build universal vocab — full run (~5 min)
# Fetches protein-coding genes from HGNC, intersects with PBMC-3K + PBMC-68K + kidney.
# Expected output: ~19,000 genes (vs 73,565 before the protein-coding filter).
import subprocess, os

TAR_PATH  = '/content/drive/MyDrive/fresh_68k_pbmc_donor_a_filtered_gene_bc_matrices.tar.gz'
VOCAB_DIR = '/content/drive/MyDrive/CellJEPA_results/universal_vocab/'

result = subprocess.run(
    ['python3', '-u', '/content/build_universal_vocab.py',
     '--tar_path',  TAR_PATH,
     '--drive_dir', VOCAB_DIR,
     '--cache_dir', '/content'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# Cell 5: Pre-training smoke test — multi-tissue, 1 epoch, ~300 cells (~5 min CPU)
# Uses 50 cells per tissue × 6 tissues = ~300 cells total.
import subprocess, os

VOCAB_FILE_SMOKE = '/content/vocab_smoke/universal_gene_names.json'

result = subprocess.run(
    ['python3', '-u', '/content/pretrain_universal.py',
     '--source',     'multitissue',
     '--vocab_file', VOCAB_FILE_SMOKE,
     '--drive_dir',  '/content/pretrain_smoke/',
     '--smoke_test', '--device', 'cpu'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# Cell 6: Pre-train SIGReg only — multi-tissue (~50k cells, 4 epochs, ~60-90 min on A100)
# Cell-JEPA checkpoint already exists on Drive — skipping it with --skip_jepa.
# Tissues: kidney, lung, liver, brain, heart, intestine (~8333 cells each)
#
# Resume from checkpoint if session disconnected:
#   add '--resume_sigreg', os.path.join(MT_DIR, 'multitissue_universal_sigreg_epochN.pt')
import subprocess, time, threading, os

VOCAB_FILE = '/content/drive/MyDrive/CellJEPA_results/universal_vocab/universal_gene_names.json'
MT_DIR     = '/content/drive/MyDrive/CellJEPA_results/multitissue_universal/'

t0   = time.time()
proc = subprocess.Popen(
    ['python3', '-u', '/content/pretrain_universal.py',
     '--source',             'multitissue',
     '--vocab_file',         VOCAB_FILE,
     '--drive_dir',          MT_DIR,
     '--n_cells_per_tissue', '8333',
     '--n_epochs',           '4',
     '--batch_size',         '32',
     '--skip_jepa',
     '--device',             'cuda'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()
rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 7: Transfer smoke test (~5 min CPU)
# Tests scratch baselines only — no checkpoints needed.
import subprocess, os

VOCAB_FILE_SMOKE = '/content/vocab_smoke/universal_gene_names.json'

result = subprocess.run(
    ['python3', '-u', '/content/run_transfer_universal.py',
     '--vocab_file',   VOCAB_FILE_SMOKE,
     '--smoke_test',   '--device', 'cpu',
     '--skip_pbmc68k',
     '--results_file', 'results_transfer_multitissue_smoke.txt'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])

In [ ]:
# Cell 8: Multi-tissue transfer → PBMC-3K
# Runs: SIGReg (multitissue pre-trained) + 2 scratch baselines
# To also run Cell-JEPA multitissue, add:
#   '--kidney_jepa_checkpoint', os.path.join(MT_DIR, 'multitissue_universal_jepa_final.pt'),
import subprocess, time, threading, os

VOCAB_FILE = '/content/drive/MyDrive/CellJEPA_results/universal_vocab/universal_gene_names.json'
MT_DIR     = '/content/drive/MyDrive/CellJEPA_results/multitissue_universal/'

t0   = time.time()
proc = subprocess.Popen(
    ['python3', '-u', '/content/run_transfer_universal.py',
     '--vocab_file',               VOCAB_FILE,
     '--kidney_sigreg_checkpoint', os.path.join(MT_DIR, 'multitissue_universal_sigreg_final.pt'),
     '--pretrain_source',          'multitissue',
     '--skip_pbmc68k',
     '--finetune_epochs', '30',
     '--pretrain_epochs', '4',
     '--device',          'cuda',
     '--results_file',    'results_transfer_multitissue.txt'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1
)

def stream(pipe):
    for line in pipe:
        print(line, end='', flush=True)

t_out = threading.Thread(target=stream, args=(proc.stdout,))
t_err = threading.Thread(target=stream, args=(proc.stderr,))
t_out.start(); t_err.start()
t_out.join(); t_err.join()
rc = proc.wait()
if rc != 0:
    print(f'\n*** PROCESS EXITED WITH CODE {rc} ***')
else:
    print(f'\nDone in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 9: Display results and plot
import os, re
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

if os.path.exists('results_transfer_multitissue.txt'):
    print(open('results_transfer_multitissue.txt').read())

def parse_universal_results(path, phase='Fine-tuned'):
    if not os.path.exists(path):
        return {}
    data = {}
    in_section = False
    for line in open(path):
        if phase in line:
            in_section = True
            continue
        if in_section and line.strip().startswith('='):
            if data:
                break
            continue
        if in_section:
            m = re.match(r'^  (.{48})\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)', line)
            if m:
                name = m.group(1).strip()
                if name and not name.startswith(('-', 'C')):
                    data[name] = {
                        'nmi': float(m.group(2)), 'ari': float(m.group(3)),
                        'asw': float(m.group(4)), 'avg_bio': float(m.group(5)),
                    }
    return data

results = parse_universal_results('results_transfer_multitissue.txt')
if results:
    conditions = list(results.keys())
    avg_bios   = [results[c]['avg_bio'] for c in conditions]

    colors  = ['#4C72B0' if 'Cell-JEPA' in c else '#DD8452' for c in conditions]
    hatches = ['' if 'scratch' in c.lower() else '///' for c in conditions]

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(conditions))
    bars = ax.bar(x, avg_bios, color=colors, alpha=0.85, width=0.6)
    for bar, hatch in zip(bars, hatches):
        bar.set_hatch(hatch)
    for xi, v in zip(x, avg_bios):
        ax.text(xi, v + 0.005, f'{v:.4f}', ha='center', fontsize=8)
    ax.set_xticks(x)
    ax.set_xticklabels(conditions, rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('AvgBIO (fine-tuned)')
    ax.set_title('Multi-tissue → PBMC-3K Transfer (Fine-tuned AvgBIO)', fontsize=11)
    ax.set_ylim(0, max(avg_bios) * 1.2 + 0.02)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)

    legend_elements = [
        Patch(facecolor='#4C72B0', label='Cell-JEPA'),
        Patch(facecolor='#DD8452', label='SIGReg'),
        Patch(facecolor='white', edgecolor='black', hatch='///', label='Multi-tissue pre-trained'),
    ]
    ax.legend(handles=legend_elements, fontsize=9)
    plt.tight_layout()
    plt.savefig('transfer_multitissue_results.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved transfer_multitissue_results.png')

In [ ]:
# Cell 10: Save results to Drive
import shutil, os

RESULTS_DIR = '/content/drive/MyDrive/CellJEPA_results/multitissue_transfer/'
files = [
    'results_transfer_multitissue.txt',
    'results_transfer_multitissue_smoke.txt',
    'transfer_multitissue_results.png',
]
for f in files:
    if os.path.exists(f):
        shutil.copy(f, RESULTS_DIR)
        print(f'Copied {f}')
    else:
        print(f'Not found: {f} (skipping)')
print(f'Done. Files in {RESULTS_DIR}')